In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
df = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

# Your work should go here

What is expected in this lab?
* Minimal data preparation
* Sklearn pipeline
* Hyperparameter tuning

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


In [6]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["Survived"].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

## splitting data for train-valid-test using stratify for unbalanced dataset

In [9]:
features = df.drop("Survived", axis=1)
label = df["Survived"]
features.shape, label.shape

((891, 11), (891,))

In [10]:
from sklearn.model_selection import train_test_split
## data embalanced. using stratify for percentage suffling across train valid test
x_train_dummy, x_test, y_train_dummy, y_test = train_test_split(features, label, test_size=0.2, random_state=42, stratify=label)
x_train, x_valid, y_train, y_valid = train_test_split(x_train_dummy, y_train_dummy, random_state=42, test_size=0.2, stratify=y_train_dummy)

In [11]:
x_train.shape, y_train.shape, x_valid.shape, y_valid.shape, x_test.shape, y_test.shape

((569, 11), (569,), (143, 11), (143,), (179, 11), (179,))

## creating pipeline and columntransform that will accept data and transform it to valid numbers to be passed to model


In [12]:
x_train.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
242,243,2,"Coleridge, Mr. Reginald Charles",male,29.0,0,0,W./C. 14263,10.5000,NaN,S
450,451,2,"West, Mr. Edwy Arthur",male,36.0,1,2,C.A. 34651,27.7500,NaN,S
721,722,3,"Jensen, Mr. Svend Lauritz",male,17.0,1,0,350048,7.0542,NaN,S
733,734,2,"Berriman, Mr. William John",male,23.0,0,0,28425,13.0000,NaN,S
191,192,2,"Carbines, Mr. William",male,19.0,0,0,28424,13.0000,NaN,S


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
num_cols = ["Age", "Fare"]
cat_nom_cols = ["SibSp", "Sex", "Embarked"]
cat_ord_cols = ["Pclass", "Parch"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_nom_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])
cat_ord_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder())
])

ct = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat_nom", cat_nom_pipeline, cat_nom_cols),
    ("cat_ord", cat_ord_pipeline, cat_ord_cols)
])

In [14]:
ct_train_data = ct.fit_transform(x_train)
ct_train_data

array([[-0.06484003, -0.45371004,  1.        , ...,  1.        ,
         1.        ,  0.        ],
       [ 0.47365714, -0.07187126,  0.        , ...,  1.        ,
         1.        ,  2.        ],
       [-0.98797805, -0.52998483,  0.        , ...,  1.        ,
         2.        ,  0.        ],
       ...,
       [-1.52647522, -0.06855092,  0.        , ...,  1.        ,
         2.        ,  2.        ],
       [ 0.39672897, -0.45371004,  1.        , ...,  1.        ,
         1.        ,  0.        ],
       [ 0.01208813, -0.45371004,  1.        , ...,  1.        ,
         1.        ,  0.        ]], shape=(569, 16))

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
ct_valid_data = ct.transform(x_valid)
LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga")
LR.fit(ct_train_data, y_train)
print(f"accuracy in training:{LR.score(ct_train_data, y_train)}\naccuracy in valid:{LR.score(ct_valid_data,y_valid)}")

accuracy in training:0.8154657293497364
accuracy in valid:0.7762237762237763


d:\anaconda\envs\iti\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## model output isnt that bad, There is room for emprovement still
## trying different hyper parameters

In [16]:
LR2 = LogisticRegression(C=100, l1_ratio=0.5, solver="saga")
LR2.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR2.score(ct_train_data, y_train)}\nValid_accuracy:{LR2.score(ct_valid_data, y_valid)}")

traininig accuracy:0.8154657293497364
Valid_accuracy:0.7762237762237763


d:\anaconda\envs\iti\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## increasing C from 50 -> 100 didnt change how model learn
## Deacreasing C->1 to see any changes

In [17]:
LR3 = LogisticRegression(C=1, l1_ratio=0.5, solver="saga")
LR3.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR3.score(ct_train_data, y_train)}\nValid_accuracy:{LR3.score(ct_valid_data, y_valid)}")

traininig accuracy:0.8154657293497364
Valid_accuracy:0.7622377622377622


d:\anaconda\envs\iti\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## model reached max iterations but didnt converge on all 3 tests
## increasing max_iteration Hyperparameter from default 100->1000 on all 3 cases

In [18]:
LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga", max_iter=1000)
LR.fit(ct_train_data, y_train)
print(f"accuracy in training:{LR.score(ct_train_data, y_train)}\naccuracy in valid:{LR.score(ct_valid_data,y_valid)}")
print("*"*100)
LR2 = LogisticRegression(C=100, l1_ratio=0.5, solver="saga", max_iter=1000)
LR2.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR2.score(ct_train_data, y_train)}\nValid_accuracy:{LR2.score(ct_valid_data, y_valid)}")
print("*"*100)
LR3 = LogisticRegression(C=1, l1_ratio=0.5, solver="saga", max_iter=1000)
LR3.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR3.score(ct_train_data, y_train)}\nValid_accuracy:{LR3.score(ct_valid_data, y_valid)}")

d:\anaconda\envs\iti\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


accuracy in training:0.8154657293497364
accuracy in valid:0.7762237762237763
****************************************************************************************************


d:\anaconda\envs\iti\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


traininig accuracy:0.8154657293497364
Valid_accuracy:0.7762237762237763
****************************************************************************************************
traininig accuracy:0.8154657293497364
Valid_accuracy:0.7622377622377622


## max_iteration still not enough

In [19]:
LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga", max_iter=3000)
LR.fit(ct_train_data, y_train)
print(f"accuracy in training:{LR.score(ct_train_data, y_train)}\naccuracy in valid:{LR.score(ct_valid_data,y_valid)}")
print("*"*100)
LR2 = LogisticRegression(C=100, l1_ratio=0.5, solver="saga", max_iter=3000)
LR2.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR2.score(ct_train_data, y_train)}\nValid_accuracy:{LR2.score(ct_valid_data, y_valid)}")
print("*"*100)
LR3 = LogisticRegression(C=1, l1_ratio=0.5, solver="saga", max_iter=3000)
LR3.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR3.score(ct_train_data, y_train)}\nValid_accuracy:{LR3.score(ct_valid_data, y_valid)}")

accuracy in training:0.8154657293497364
accuracy in valid:0.7762237762237763
****************************************************************************************************
traininig accuracy:0.8154657293497364
Valid_accuracy:0.7762237762237763
****************************************************************************************************
traininig accuracy:0.8154657293497364
Valid_accuracy:0.7622377622377622


## applying early stopping using tol hyperparameter from default 1e-4 to 1e-2

In [20]:
LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga", max_iter=3000, tol=1e-2)
LR.fit(ct_train_data, y_train)
print(f"accuracy in training:{LR.score(ct_train_data, y_train)}\naccuracy in valid:{LR.score(ct_valid_data,y_valid)}")
print("*"*100)
LR2 = LogisticRegression(C=100, l1_ratio=0.5, solver="saga", max_iter=3000, tol=1e-2)
LR2.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR2.score(ct_train_data, y_train)}\nValid_accuracy:{LR2.score(ct_valid_data, y_valid)}")
print("*"*100)
LR3 = LogisticRegression(C=1, l1_ratio=0.5, solver="saga", max_iter=3000, tol=1e-2)
LR3.fit(ct_train_data, y_train)
print(f"traininig accuracy:{LR3.score(ct_train_data, y_train)}\nValid_accuracy:{LR3.score(ct_valid_data, y_valid)}")

accuracy in training:0.8172231985940246
accuracy in valid:0.7622377622377622
****************************************************************************************************
traininig accuracy:0.8172231985940246
Valid_accuracy:0.7622377622377622
****************************************************************************************************
traininig accuracy:0.8084358523725835
Valid_accuracy:0.7622377622377622


## polynomial degree  testing

In [21]:
from sklearn.preprocessing import PolynomialFeatures
poly_d2 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
ct_polyd2_train_data = poly_d2.fit_transform(ct_train_data)
ct_polyd2_valid_data = poly_d2.transform(ct_valid_data)
poly_LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR.fit(ct_polyd2_train_data, y_train)
print(f"accuracy in training:{poly_LR.score(ct_polyd2_train_data, y_train)}\naccuracy in valid:{poly_LR.score(ct_polyd2_valid_data,y_valid)}")
print("*"*100)
poly_LR2 = LogisticRegression(C=100, l1_ratio=0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR2.fit(ct_polyd2_train_data, y_train)
print(f"traininig accuracy:{poly_LR2.score(ct_polyd2_train_data, y_train)}\nValid_accuracy:{poly_LR2.score(ct_polyd2_valid_data, y_valid)}")
print("*"*100)
poly_LR3 = LogisticRegression(C=1, l1_ratio=0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR3.fit(ct_polyd2_train_data, y_train)
print(f"traininig accuracy:{poly_LR3.score(ct_polyd2_train_data, y_train)}\nValid_accuracy:{poly_LR3.score(ct_polyd2_valid_data, y_valid)}")

accuracy in training:0.8400702987697716
accuracy in valid:0.8391608391608392
****************************************************************************************************
traininig accuracy:0.8400702987697716
Valid_accuracy:0.8391608391608392
****************************************************************************************************
traininig accuracy:0.8400702987697716
Valid_accuracy:0.8321678321678322


## improving poly to degree 2 got better accuracy in train and validation
## increasing it more to see how far can it reach

In [22]:
poly_d3 = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)
ct_polyd3_train_data = poly_d3.fit_transform(ct_train_data)
ct_polyd3_valid_data = poly_d3.transform(ct_valid_data)
poly_LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR.fit(ct_polyd3_train_data, y_train)
print(f"accuracy in training:{poly_LR.score(ct_polyd3_train_data, y_train)}\naccuracy in valid:{poly_LR.score(ct_polyd3_valid_data,y_valid)}")
print("*"*100)
poly_LR2 = LogisticRegression(C=100, l1_ratio=0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR2.fit(ct_polyd3_train_data, y_train)
print(f"traininig accuracy:{poly_LR2.score(ct_polyd3_train_data, y_train)}\nValid_accuracy:{poly_LR2.score(ct_polyd3_valid_data, y_valid)}")
print("*"*100)
poly_LR3 = LogisticRegression(C=1, l1_ratio=0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR3.fit(ct_polyd3_train_data, y_train)
print(f"traininig accuracy:{poly_LR3.score(ct_polyd3_train_data, y_train)}\nValid_accuracy:{poly_LR3.score(ct_polyd3_valid_data, y_valid)}")

accuracy in training:0.8330404217926186
accuracy in valid:0.8181818181818182
****************************************************************************************************
traininig accuracy:0.8347978910369068
Valid_accuracy:0.8111888111888111
****************************************************************************************************
traininig accuracy:0.8330404217926186
Valid_accuracy:0.8111888111888111


## less accuracy in poly=3 meaning data fits better on equations from second degree

# Trying best model on testing data after hyperparameters tuning

In [23]:
ct_test_data = ct.transform(x_test)
ct_polyd2_test_data = poly_d2.transform(ct_test_data)
poly_LR = LogisticRegression(C=50, l1_ratio= 0.5, solver="saga", max_iter=3000, tol=1e-2)
poly_LR.fit(ct_polyd2_train_data, y_train)
print(f"Acc For training:{poly_LR.score(ct_polyd2_train_data, y_train)}\nAcc For Valid:{poly_LR.score(ct_polyd2_valid_data, y_valid)}")
print(f"Acc for Testing:{poly_LR.score(ct_polyd2_test_data, y_test)}")

Acc For training:0.8400702987697716
Acc For Valid:0.8391608391608392
Acc for Testing:0.7821229050279329


## Trying different model "SVC"

In [24]:
from sklearn.svm import LinearSVC
SVC = LinearSVC(C=100, max_iter=3000, penalty="l2")
SVC2 = LinearSVC(C=50, max_iter=3000, penalty="l2")
SVC3 = LinearSVC(C=1, max_iter=3000, penalty="l2")
SVC4 = LinearSVC(C=100, max_iter=3000, penalty="l1")
SVC5 = LinearSVC(C=50, max_iter=3000, penalty="l1")
SVC6 = LinearSVC(C=1, max_iter=3000, penalty="l1")
SVC.fit(ct_train_data, y_train)
SVC2.fit(ct_train_data, y_train)
SVC3.fit(ct_train_data, y_train)
SVC4.fit(ct_train_data, y_train)
SVC5.fit(ct_train_data, y_train)
SVC6.fit(ct_train_data, y_train)
print(SVC.score(ct_train_data, y_train),SVC.score(ct_valid_data, y_valid))
print(SVC2.score(ct_train_data, y_train),SVC2.score(ct_valid_data, y_valid))
print(SVC3.score(ct_train_data, y_train),SVC3.score(ct_valid_data, y_valid))
print(SVC4.score(ct_train_data, y_train),SVC4.score(ct_valid_data, y_valid))
print(SVC5.score(ct_train_data, y_train),SVC5.score(ct_valid_data, y_valid))
print(SVC6.score(ct_train_data, y_train),SVC6.score(ct_valid_data, y_valid))

0.8154657293497364 0.7692307692307693
0.8154657293497364 0.7692307692307693
0.8154657293497364 0.7692307692307693
0.8154657293497364 0.7692307692307693
0.8154657293497364 0.7692307692307693
0.8137082601054482 0.7692307692307693


d:\anaconda\envs\iti\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## C and penality doesnt effect the model much.
## trying the poly with degree 2-3

In [25]:
from sklearn.svm import SVC
SVC_poly2 = SVC(kernel="poly", degree=2,C=100, max_iter=3000)
SVC2_poly2 = SVC(kernel="poly", degree=2,C=50, max_iter=3000)
SVC3_poly2 = SVC(kernel="poly", degree=2,C=1, max_iter=3000)
SVC4_poly3 = SVC(kernel="poly", degree=3,C=100, max_iter=3000)
SVC5_poly3 = SVC(kernel="poly", degree=3,C=50, max_iter=3000)
SVC6_poly3 = SVC(kernel="poly", degree=3,C=1, max_iter=3000)
SVC_poly2.fit(ct_train_data, y_train)
SVC2_poly2.fit(ct_train_data, y_train)
SVC3_poly2.fit(ct_train_data, y_train)
SVC4_poly3.fit(ct_train_data, y_train)
SVC5_poly3.fit(ct_train_data, y_train)
SVC6_poly3.fit(ct_train_data, y_train)
print(SVC_poly2.score(ct_train_data, y_train),SVC_poly2.score(ct_valid_data, y_valid))
print(SVC2_poly2.score(ct_train_data, y_train),SVC2_poly2.score(ct_valid_data, y_valid))
print(SVC3_poly2.score(ct_train_data, y_train),SVC3_poly2.score(ct_valid_data, y_valid))
print(SVC4_poly3.score(ct_train_data, y_train),SVC4_poly3.score(ct_valid_data, y_valid))
print(SVC5_poly3.score(ct_train_data, y_train),SVC5_poly3.score(ct_valid_data, y_valid))
print(SVC6_poly3.score(ct_train_data, y_train),SVC6_poly3.score(ct_valid_data, y_valid))

0.8541300527240774 0.8251748251748252
0.8506151142355008 0.8251748251748252
0.827768014059754 0.8391608391608392
0.8681898066783831 0.8041958041958042
0.8681898066783831 0.8041958041958042
0.8471001757469244 0.8321678321678322


d:\anaconda\envs\iti\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
d:\anaconda\envs\iti\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
d:\anaconda\envs\iti\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
d:\anaconda\envs\iti\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [26]:
SVC3_poly2.score(ct_test_data, y_test)

0.8156424581005587

## model accuracy on test data is okay

In [27]:
best_svc_model = SVC3_poly2.fit(ct_train_data, y_train)
best_LR_model = poly_LR2.fit(ct_polyd2_train_data, y_train)

In [28]:
best_svc_model , best_LR_model

(SVC(C=1, degree=2, kernel='poly', max_iter=3000),
 LogisticRegression(C=100, l1_ratio=0.5, max_iter=3000, solver='saga', tol=0.01))

In [29]:
import joblib
file_name_svc = "svc_poly2_model"
file_name_lr = "lr_poly2_model"
joblib.dump(best_svc_model, file_name_svc)
joblib.dump(best_LR_model, file_name_lr)

['lr_poly2_model']

In [30]:
import joblib
file_name_lr = "lr_poly2_model"
file_name_svc = "svc_poly2_model"
load_lr_model = joblib.load(file_name_lr)
load_svc_model = joblib.load(file_name_svc)

In [31]:
load_lr_model,load_svc_model

(LogisticRegression(C=100, l1_ratio=0.5, max_iter=3000, solver='saga', tol=0.01),
 SVC(C=1, degree=2, kernel='poly', max_iter=3000))

## testing inference time

In [32]:
import time
%timeit load_lr_model.predict(ct_polyd2_test_data[:1])
%timeit load_svc_model.predict(ct_test_data[:1])

43.2 μs ± 2.31 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
66.4 μs ± 2.07 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [33]:
%timeit load_lr_model.predict(ct_polyd2_test_data)
%timeit load_svc_model.predict(ct_test_data)

49.6 μs ± 457 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
498 μs ± 16.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Continuing on Task1 From here

In [35]:
num_cols = ["Age", "Fare"]
cat_nom_cols = ["SibSp", "Sex", "Embarked"]
cat_ord_cols = ["Pclass", "Parch"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_nom_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])
cat_ord_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder())
])

ct = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat_nom", cat_nom_pipeline, cat_nom_cols),
    ("cat_ord", cat_ord_pipeline, cat_ord_cols)
])

In [38]:
x_encoded_train = ct.fit_transform(x_train)
x_encoded_valid = ct.transform(x_valid)
x_encoded_test = ct.transform(x_test)

In [40]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}
rf = RandomForestClassifier(random_state=42)
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, random_state=42)
rf.fit(x_encoded_train, y_train)
rf_score_train = rf.score(x_encoded_train, y_train)
rf_score_valid = rf.score(x_encoded_valid, y_valid)
print(f"random forest training score: {rf_score_train}\nrandom forest valid score: {rf_score_valid}")

rf_random.fit(x_encoded_train, y_train)
best_rf_model = rf_random.best_estimator_
best_rf_score_train = best_rf_model.score(x_encoded_train, y_train)
best_rf_score_valid = best_rf_model.score(x_encoded_valid, y_valid)
print(f"best random forest training score: {best_rf_score_train}\nbest random forest valid score: {best_rf_score_valid}")
print(f"best parameters: {rf_random.best_params_}")




random forest training score: 0.9806678383128296
random forest valid score: 0.8321678321678322
best random forest training score: 0.9402460456942003
best random forest valid score: 0.8251748251748252
best parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10, 'bootstrap': False}


In [48]:
rf_test_score = rf.score(x_encoded_test, y_test)
rf_random_test_score = best_rf_model.score(x_encoded_test, y_test)
print(f"Random Forest Test Score: {rf_test_score}")
print(f"Best Random Forest Test Score: {rf_random_test_score}")

Random Forest Test Score: 0.7597765363128491
Best Random Forest Test Score: 0.7932960893854749


In [43]:
rf_random_df = pd.DataFrame(rf_random.cv_results_)
rf_random_df.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_max_depth,param_bootstrap,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.053465,0.011227,0.004665,0.001675,50,5,2,30,False,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.798246,0.798246,0.815789,0.824561,0.805310,0.808430,0.010311,5
1,0.039009,0.001493,0.003857,0.000233,50,10,4,30,False,"{'n_estimators': 50, 'min_samples_split': 10, ...",0.798246,0.815789,0.833333,0.833333,0.805310,0.817202,0.014305,2
2,0.037152,0.001045,0.003551,0.000334,50,5,1,10,False,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.798246,0.780702,0.824561,0.842105,0.823009,0.813725,0.021630,4
3,0.035527,0.000685,0.003359,0.000091,50,10,2,20,False,"{'n_estimators': 50, 'min_samples_split': 10, ...",0.807018,0.789474,0.824561,0.815789,0.787611,0.804891,0.014467,8
4,0.042348,0.000500,0.003322,0.000130,50,10,2,None,True,"{'n_estimators': 50, 'min_samples_split': 10, ...",0.798246,0.789474,0.807018,0.815789,0.823009,0.806707,0.011974,6


In [44]:
rf_random_df = rf_random_df.sort_values(by="rank_test_score")
rf_random_df.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_max_depth,param_bootstrap,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
9,0.071700,0.001589,0.005930,0.000201,100,5,1,10,False,"{'n_estimators': 100, 'min_samples_split': 5, ...",0.789474,0.789474,0.842105,0.842105,0.831858,0.819003,0.024399,1
1,0.039009,0.001493,0.003857,0.000233,50,10,4,30,False,"{'n_estimators': 50, 'min_samples_split': 10, ...",0.798246,0.815789,0.833333,0.833333,0.805310,0.817202,0.014305,2
8,0.071493,0.001573,0.005995,0.000226,100,10,2,30,False,"{'n_estimators': 100, 'min_samples_split': 10,...",0.815789,0.807018,0.824561,0.833333,0.796460,0.815432,0.012920,3
2,0.037152,0.001045,0.003551,0.000334,50,5,1,10,False,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.798246,0.780702,0.824561,0.842105,0.823009,0.813725,0.021630,4
0,0.053465,0.011227,0.004665,0.001675,50,5,2,30,False,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.798246,0.798246,0.815789,0.824561,0.805310,0.808430,0.010311,5


In [46]:
rf_random_df.columns

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_n_estimators', 'param_min_samples_split',
       'param_min_samples_leaf', 'param_max_depth', 'param_bootstrap',
       'params', 'split0_test_score', 'split1_test_score', 'split2_test_score',
       'split3_test_score', 'split4_test_score', 'mean_test_score',
       'std_test_score', 'rank_test_score'],
      dtype='object')

In [47]:
cols_keep = ["params", "mean_test_score", "rank_test_score"]
rf_random_df_filtered = rf_random_df[cols_keep]
rf_random_df_filtered.head()

,params,mean_test_score,rank_test_score
9,"{'n_estimators': 100, 'min_samples_split': 5, ...",0.819003,1
1,"{'n_estimators': 50, 'min_samples_split': 10, ...",0.817202,2
8,"{'n_estimators': 100, 'min_samples_split': 10,...",0.815432,3
2,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.813725,4
0,"{'n_estimators': 50, 'min_samples_split': 5, '...",0.808430,5


## Random forest best params is overfitting(train acc: 0.94%, valid score: 82%)
## perform poorly on the test set even worse than the SVC model with (test score: 79%) yet its better than LR model

## why did model overfit?
## i would assume because of the random search space didnt give model the chance to be simpler so it overfitting on the training data

In [50]:
%timeit best_rf_model.predict(x_encoded_test)
%timeit best_rf_model.predict(x_encoded_test[:1])

5.92 ms ± 171 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.08 ms ± 167 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## SUMMARY:
## splitting data to train-valid-test with percentage 0.6-0.2-0.2
## using stratify to balance the labels across all splitted data
## trying diff models with diff hyperparameters to see who fits best on valid data then using it on test data for end goal test
## Logistec regression model:
* Changing C hyperparameter and penality didnt effect how model perform on train-valid data. accuracy in training:0.81722 accuracy in testing:0.7622

* applying polynomial 2 increased model accuracy on training and valid.traininig accuracy:0.84007 Valid_accuracy:0.8391

* applying poly = 3 made model worse on training and valid data. traininig accuracy:0.83304 Valid_accuracy:0.81118

* best model Acc for Testing:0.7821229050279329

## SVC model:
* Same as LogistecRegression model the C and penality didnt effect model performance

* Acc in training and validation : 0.8154657293497364 0.7692307692307693

* Changing Kernel to increase model complexity to degree 2-3.
* Best model on degree 2 with C=1: 0.827768014059754  0.8391608391608392
* Best model on degree 3 with C=1: 0.8471001757469244 0.8321678321678322
* Acc of best model (degree2) on test data: 0.8156424581005587

## RF classifier:
* Random forest best params is overfitting(train acc: 0.94%, valid score: 82%)
* perform poorly on the test set even worse than the SVC model with (test score: 79%) yet its better than LR model

## SVC generalized better than LR model with diff of 3.5% accuracy but inference time of LR model is faster than SVC model in single row(LR:44.7 μs ± 1.83 ,SVC:62.6 μs ± 2.05), and for fulldata(LR:49.6 μs ± 1.96 μs ,SVC:487 μs ± 5.22 μs)

## Inference time of RF on all data: 5.92 ms, on single row: 5.08 ms. Which is way quicker on all data and single data than both SVC and LR

## Overall 2 models are good since they are better than the baseline estimator (61.62%)


In [34]:
# Remove this class and replace it with your work.

from sklearn.base import BaseEstimator

class DummyEstimator(BaseEstimator):
    def __init__(self, dummy_label=0):
        self.dummy_label = dummy_label
    
    def predict(self, X, y=None):
        return np.asarray([self.dummy_label] * len(X))
    
clf = DummyEstimator()

In [55]:
preds = clf.predict(df)
preds.shape

(891,)

In [56]:
from sklearn.metrics import accuracy_score
base_accuracy = accuracy_score(df["Survived"], preds)
base_accuracy

0.6161616161616161

# Submission

In [50]:
df_submission = df_test.copy()
df_submission["Survived"] = preds